# Flexibility against temperature — experiments

Exploration of flexibility as a function of outdoor temperature for *Estimating
the Installed Capacity of Aggregated Thermostatically Controlled Loads from Net
Load and Temperature*. Each step calls one function of `scripts/flex_temperature.py`.
Step reports, expectations and deviations are in `FLEX_LOG.md`; figures go to
`figures/flex_temperature/`.

Every step pickles its results under `results/flex_temperature/`. With its
`REBUILD_*` flag at `False`, a step reloads them instead of recomputing.

In [1]:
import os, sys
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, 'paper')) and os.path.dirname(_root) != _root:
    _root = os.path.dirname(_root)
os.chdir(_root)
for _p in ('scripts', 'src'):
    if os.path.abspath(_p) not in sys.path:
        sys.path.insert(0, os.path.abspath(_p))

import pandas as pd
import flex_temperature as FT
pd.set_option('display.width', 200)

# control flags: True recomputes the step and overwrites its pickles
REBUILD_STEP0 = False   # rebuilds the 1260-aggregate design and its hourly flexibility (about 15 s)
REBUILD_STEP1 = False   # rebuilds the per-aggregate daily SF and hourly flexibility (about 15 s)
print('repo root:', _root)

repo root: C:\Users\VENANCIA\OneDrive - VITO\Desktop\hp-sensitivity-paper


## Step 0 — Reproduce the current flexibility results

Rebuilds the published 20-split Kloten design (seed 0, 1260 aggregates) and its
hourly-reference actual flexibility, and checks every Section IV-F number of the
draft against its tolerance.

In [2]:
checks, step0 = FT.reproduce_flex(rebuild=REBUILD_STEP0)
print('rebuilt design identical to the paper cache:', step0['same_as_paper_cache'])
checks

rebuilt design identical to the paper cache: True


,quantity,draft,reproduced,tol,ok
0,flex_ratio_median,1.040,1.042531,0.0200,True
1,flex_ratio_q25,0.970,0.973425,0.0200,True
2,flex_ratio_q75,1.170,1.167233,0.0200,True
3,flex_wape_pct,10.000,9.874580,1.0000,True
4,energy_factor_median,0.970,0.968235,0.0200,True
5,unused_cap_factor_median,1.070,1.069075,0.0200,True
6,corr_flex_energy,0.960,0.960143,0.0200,True
7,dailymean_over_hourly_actual_pct,3.000,3.406694,1.0000,True
8,ratio_vs_dailymean_actual,1.010,1.005890,0.0200,True
9,deadband_floor_share_pct,9.000,9.034939,1.0000,True


In [3]:
step0['context']

{'T_min': -8.24843692779541,
 'T_max': 26.532812118530273,
 'n_days': 365,
 'T_cross_pilot0_with_b': -9.01684515089432,
 'T_cross_fig11_estimate': -10.183939040650085,
 'n_days_below_fig11_cross': 0,
 'n_heating_days_fig11': 246,
 'T_h_target_median': 16.528659714980407,
 'T_h_target_iqr': (16.238221659843187, 16.96571254972177),
 'pilot_m_median': 0.017841154580689644,
 'pilot_m_iqr': (0.017320816228591305, 0.01868294410849839),
 'fig11_sf_e_max': 0.4637562036514282}

## Step 1 — Headroom and flexible energy against temperature

Per aggregate and 1 °C bin of heating-regime days: P_down, P_up and F under W1
(constant window, per hour of window) and W2 (Δt ∝ 1/(T_in − T), normalised), for
the estimate and the actual, per kW of true installed capacity. Primary setting
N = 50, N_hp = 25; repeated at N = 10 and 30, and with the pilot intercept b_h
added to the estimate (diagnostic only). Expectations are in `FLEX_LOG.md`.

In [4]:
step1 = FT.step1(rebuild=REBUILD_STEP1)
P = step1[FT.PRIMARY]
print(P['fig'])
for k, v in P['checks'].items():
    print(k, v)
P['peaks'].round(2)

figures\flex_temperature\fig_flex_temperature
E1.1 estimate (True, 0.0347318804502279)
E1.1 actual (True, 0.0026523904574924373)
E1.2a T_in=20 (False, 10.39)
E1.2b T_in=20 (True, 5.78)
E1.2a T_in=21 (True, 7.9)
E1.2b T_in=21 (True, 4.67)
E1.3 estimate (True, 0.0)
E1.3 actual (True, 0.0)
E1.4 (False, (0.2618488524455578, 1.1662540644327466))
E1.5 estimate (True, 0.9339077174406132)
E1.5 actual (False, 5.301046130403959)
E1.6 (True, [0.69, 0.35, 0.26])
E1.7 (True, 0.7628883213400321)


,window,curve,peak_split_med,peak_split_q25,peak_split_q75,peak_split_iqr,cold_over_peak
0,W1,estimate,-3.84,-3.84,-3.84,0.00,1.00
1,W1,actual,-3.84,-3.84,-3.84,0.00,1.00
2,"W2, T_in = 20 °C",estimate,5.78,5.14,6.08,0.93,0.89
3,"W2, T_in = 20 °C",actual,10.39,10.30,15.60,5.30,0.76
4,"W2, T_in = 21 °C",estimate,4.67,4.16,4.95,0.80,0.91
5,"W2, T_in = 21 °C",actual,7.90,7.85,8.01,0.16,0.81


In [5]:
P['bins']

,bin (°C),days,aggregates,F est,F act,est/act (aggregates),est/act (splits),P_up est/act,energy,headroom,hourly
0,"[-9, -2)",7,60,"0.263 [0.239, 0.278]","0.226 [0.223, 0.229]","1.17 [1.07, 1.22]","1.15 [1.05, 1.23]",1.17,1.09,1.02,1.04
1,"[-2, 0)",13,60,"0.242 [0.221, 0.256]","0.217 [0.215, 0.220]","1.11 [1.04, 1.16]","1.10 [1.02, 1.17]",1.21,1.01,1.06,1.03
2,"[0, 1)",13,60,"0.234 [0.214, 0.247]","0.215 [0.211, 0.219]","1.08 [1.02, 1.13]","1.07 [1.01, 1.14]",1.22,0.99,1.06,1.02
3,"[1, 2)",10,60,"0.222 [0.204, 0.235]","0.205 [0.202, 0.208]","1.08 [1.02, 1.13]","1.07 [1.01, 1.14]",1.21,0.99,1.06,1.03
4,"[2, 3)",11,60,"0.214 [0.197, 0.227]","0.194 [0.191, 0.199]","1.09 [1.03, 1.15]","1.08 [1.02, 1.15]",1.20,1.01,1.04,1.03
5,"[3, 4)",11,60,"0.205 [0.188, 0.216]","0.186 [0.182, 0.190]","1.09 [1.03, 1.14]","1.08 [1.02, 1.15]",1.20,1.00,1.04,1.04
6,"[4, 5)",14,60,"0.193 [0.178, 0.203]","0.179 [0.173, 0.182]","1.07 [1.02, 1.12]","1.07 [1.01, 1.12]",1.19,1.00,1.03,1.03
7,"[5, 6)",14,60,"0.181 [0.167, 0.190]","0.170 [0.164, 0.172]","1.05 [1.01, 1.11]","1.05 [1.01, 1.12]",1.19,0.99,1.03,1.03
8,"[6, 7)",15,60,"0.168 [0.155, 0.177]","0.153 [0.149, 0.156]","1.08 [1.04, 1.14]","1.08 [1.04, 1.14]",1.17,1.02,1.02,1.03
9,"[7, 8)",29,60,"0.156 [0.144, 0.165]","0.153 [0.147, 0.156]","1.00 [0.96, 1.06]","1.00 [0.96, 1.07]",1.19,0.94,1.04,1.02


In [6]:
for k in [(30, 15), (10, 5), 'primary_with_b']:
    print('=====', k)
    display(step1[k]['peaks'].round(2), step1[k]['bins'])

===== (30, 15)


,window,curve,peak_split_med,peak_split_q25,peak_split_q75,peak_split_iqr,cold_over_peak
0,W1,estimate,-3.84,-3.84,-3.84,0.00,1.00
1,W1,actual,-3.84,-3.84,-3.84,0.00,1.00
2,"W2, T_in = 20 °C",estimate,5.46,4.93,5.94,1.00,0.88
3,"W2, T_in = 20 °C",actual,10.37,10.22,15.60,5.38,0.75
4,"W2, T_in = 21 °C",estimate,4.47,3.95,4.90,0.95,0.90
5,"W2, T_in = 21 °C",actual,7.92,7.82,10.28,2.46,0.80


,bin (°C),days,aggregates,F est,F act,est/act (aggregates),est/act (splits),P_up est/act,energy,headroom,hourly
0,"[-9, -2)",7,60,"0.265 [0.245, 0.283]","0.225 [0.219, 0.229]","1.17 [1.11, 1.24]","1.15 [1.12, 1.22]",1.17,1.08,1.02,1.04
1,"[-2, 0)",13,60,"0.245 [0.226, 0.261]","0.217 [0.212, 0.222]","1.12 [1.07, 1.19]","1.10 [1.07, 1.16]",1.22,0.99,1.05,1.04
2,"[0, 1)",13,60,"0.237 [0.219, 0.252]","0.215 [0.210, 0.221]","1.09 [1.04, 1.16]","1.09 [1.05, 1.13]",1.22,0.98,1.06,1.03
3,"[1, 2)",10,60,"0.225 [0.208, 0.240]","0.205 [0.198, 0.211]","1.09 [1.04, 1.15]","1.09 [1.04, 1.12]",1.22,0.98,1.05,1.03
4,"[2, 3)",11,60,"0.217 [0.200, 0.231]","0.195 [0.188, 0.202]","1.10 [1.04, 1.15]","1.10 [1.05, 1.13]",1.20,1.00,1.04,1.04
5,"[3, 4)",11,60,"0.207 [0.191, 0.221]","0.186 [0.179, 0.194]","1.10 [1.04, 1.14]","1.10 [1.06, 1.12]",1.20,0.99,1.05,1.04
6,"[4, 5)",14,60,"0.194 [0.180, 0.208]","0.178 [0.170, 0.186]","1.08 [1.03, 1.12]","1.09 [1.05, 1.10]",1.19,1.00,1.04,1.03
7,"[5, 6)",14,60,"0.183 [0.169, 0.195]","0.167 [0.160, 0.176]","1.06 [1.03, 1.11]","1.07 [1.04, 1.09]",1.19,0.98,1.04,1.03
8,"[6, 7)",15,60,"0.170 [0.157, 0.181]","0.152 [0.145, 0.159]","1.09 [1.05, 1.16]","1.10 [1.05, 1.12]",1.18,1.02,1.03,1.04
9,"[7, 8)",29,60,"0.157 [0.146, 0.169]","0.152 [0.145, 0.160]","1.01 [0.97, 1.07]","1.01 [0.98, 1.05]",1.20,0.94,1.05,1.03


===== (10, 5)


,window,curve,peak_split_med,peak_split_q25,peak_split_q75,peak_split_iqr,cold_over_peak
0,W1,estimate,-3.84,-3.84,-3.84,0.00,1.00
1,W1,actual,-3.84,-3.84,-3.84,0.00,1.00
2,"W2, T_in = 20 °C",estimate,5.26,4.82,6.23,1.41,0.89
3,"W2, T_in = 20 °C",actual,10.47,8.39,15.60,7.20,0.71
4,"W2, T_in = 21 °C",estimate,4.27,3.88,5.04,1.15,0.91
5,"W2, T_in = 21 °C",actual,8.38,7.93,10.12,2.19,0.78


,bin (°C),days,aggregates,F est,F act,est/act (aggregates),est/act (splits),P_up est/act,energy,headroom,hourly
0,"[-9, -2)",7,60,"0.245 [0.219, 0.306]","0.217 [0.206, 0.223]","1.14 [1.05, 1.37]","1.19 [1.05, 1.29]",1.06,1.07,0.99,1.05
1,"[-2, 0)",13,60,"0.228 [0.207, 0.280]","0.207 [0.197, 0.219]","1.12 [1.02, 1.29]","1.14 [1.04, 1.24]",1.10,1.00,1.03,1.05
2,"[0, 1)",13,60,"0.221 [0.201, 0.270]","0.207 [0.193, 0.221]","1.11 [1.01, 1.22]","1.12 [1.02, 1.20]",1.08,0.98,1.04,1.05
3,"[1, 2)",10,60,"0.211 [0.191, 0.256]","0.193 [0.183, 0.213]","1.10 [1.01, 1.23]","1.09 [1.04, 1.20]",1.09,0.99,1.03,1.05
4,"[2, 3)",11,60,"0.203 [0.185, 0.246]","0.187 [0.176, 0.201]","1.11 [1.03, 1.23]","1.12 [1.04, 1.22]",1.08,1.00,1.02,1.05
5,"[3, 4)",11,60,"0.194 [0.176, 0.234]","0.178 [0.168, 0.193]","1.11 [1.02, 1.23]","1.12 [1.05, 1.22]",1.09,1.00,1.03,1.06
6,"[4, 5)",14,60,"0.183 [0.165, 0.220]","0.172 [0.157, 0.188]","1.10 [1.02, 1.19]","1.10 [1.06, 1.18]",1.08,0.99,1.02,1.05
7,"[5, 6)",14,60,"0.172 [0.154, 0.205]","0.162 [0.146, 0.180]","1.09 [1.00, 1.18]","1.09 [1.06, 1.17]",1.09,0.98,1.02,1.05
8,"[6, 7)",15,60,"0.161 [0.143, 0.191]","0.145 [0.132, 0.163]","1.12 [1.04, 1.20]","1.14 [1.09, 1.20]",1.07,1.03,1.01,1.06
9,"[7, 8)",29,60,"0.150 [0.132, 0.179]","0.146 [0.132, 0.164]","1.04 [0.96, 1.10]","1.04 [1.02, 1.14]",1.09,0.93,1.03,1.05


===== primary_with_b


,window,curve,peak_split_med,peak_split_q25,peak_split_q75,peak_split_iqr,cold_over_peak
0,W1,estimate,-3.84,-3.84,-3.84,0.00,1.00
1,W1,actual,-3.84,-3.84,-3.84,0.00,1.00
2,"W2, T_in = 20 °C",estimate,9.66,8.95,10.27,1.33,0.80
3,"W2, T_in = 20 °C",actual,10.39,10.30,15.60,5.30,0.76
4,"W2, T_in = 21 °C",estimate,7.90,7.46,8.67,1.21,0.84
5,"W2, T_in = 21 °C",actual,7.90,7.85,8.01,0.16,0.81


,bin (°C),days,aggregates,F est,F act,est/act (aggregates),est/act (splits),P_up est/act,energy,headroom,hourly
0,"[-9, -2)",7,60,"0.272 [0.245, 0.287]","0.226 [0.223, 0.229]","1.21 [1.09, 1.26]","1.18 [1.07, 1.27]",1.12,1.17,0.97,1.04
1,"[-2, 0)",13,60,"0.255 [0.230, 0.269]","0.217 [0.215, 0.220]","1.17 [1.07, 1.22]","1.15 [1.06, 1.22]",1.15,1.10,1.01,1.03
2,"[0, 1)",13,60,"0.248 [0.224, 0.261]","0.215 [0.211, 0.219]","1.14 [1.06, 1.19]","1.13 [1.05, 1.20]",1.17,1.08,1.01,1.02
3,"[1, 2)",10,60,"0.237 [0.215, 0.250]","0.205 [0.202, 0.208]","1.15 [1.06, 1.21]","1.14 [1.06, 1.20]",1.16,1.09,1.01,1.03
4,"[2, 3)",11,60,"0.229 [0.208, 0.243]","0.194 [0.191, 0.199]","1.17 [1.09, 1.23]","1.16 [1.08, 1.24]",1.15,1.12,0.99,1.03
5,"[3, 4)",11,60,"0.220 [0.200, 0.235]","0.186 [0.182, 0.190]","1.17 [1.10, 1.24]","1.16 [1.09, 1.25]",1.15,1.11,0.99,1.04
6,"[4, 5)",14,60,"0.209 [0.190, 0.224]","0.179 [0.173, 0.182]","1.17 [1.10, 1.23]","1.15 [1.10, 1.24]",1.14,1.13,1.00,1.03
7,"[5, 6)",14,60,"0.198 [0.180, 0.212]","0.170 [0.164, 0.172]","1.16 [1.11, 1.24]","1.15 [1.10, 1.25]",1.14,1.13,1.00,1.03
8,"[6, 7)",15,60,"0.187 [0.169, 0.200]","0.153 [0.149, 0.156]","1.21 [1.15, 1.28]","1.20 [1.15, 1.29]",1.13,1.19,0.99,1.03
9,"[7, 8)",29,60,"0.176 [0.159, 0.188]","0.153 [0.147, 0.156]","1.14 [1.08, 1.21]","1.13 [1.08, 1.22]",1.15,1.10,1.01,1.02
